# MalthusJAX Level 1 Demo: Core Components

This notebook demonstrates the foundational **Level 1 components** of MalthusJAX, showcasing the abstract class architecture that enables flexible genome representations and fitness evaluation.

## Overview

Level 1 provides the core abstractions and implementations for:
- **Genome representations**: Binary, Real-valued, Categorical, and Linear genomes
- **Population structures**: Type-safe, vectorized population containers
- **Fitness evaluators**: Pluggable evaluation framework with automatic batching

## Key Architecture Features

1. **Abstract Base Classes**: `BaseGenome`, `BasePopulation[G]`, and `BaseEvaluator[G, C, D]` provide type-safe interfaces
2. **Immutable Structures**: All components use `@struct.dataclass` for JAX compatibility
3. **Automatic Vectorization**: Population-level operations use `jax.vmap()` internally
4. **Configuration Separation**: Static configuration objects enable JIT compilation
5. **Pluggable Evaluation**: Evaluators accept arbitrary genome types through generics

This demo will walk through each genome type, demonstrate population operations, and showcase different fitness evaluators to illustrate the flexibility of the abstract class design.

In [14]:
# Import required libraries
import jax
import jax.numpy as jnp
import jax.random as jar
import malthusjax as mjx
from malthusjax.core.genome.binary_genome import BinaryGenome, BinaryGenomeConfig, BinaryPopulation
from malthusjax.core.genome.real_genome import RealGenome, RealGenomeConfig, RealPopulation
from malthusjax.core.genome.categorical_genome import CategoricalGenome, CategoricalGenomeConfig, CategoricalPopulation
from malthusjax.core.genome.linear import LinearGenome, LinearGenomeConfig, LinearPopulation
from malthusjax.core.fitness.binary_evaluators import BinarySumEvaluator, BinarySumConfig, KnapsackEvaluator, KnapsackConfig
from malthusjax.core.fitness.real_evaluators import SphereEvaluator, SphereConfig, GriewankEvaluator, GriewankConfig
from malthusjax.core.fitness.linear_gp_evaluator import LinearGPEvaluator
import time

print(f"MalthusJAX Version: {mjx.__version__}")
print(f"JAX Backend: {jax.default_backend()}")
print("Level 1 components loaded successfully")

# Initialize random key for reproducibility
key = jar.PRNGKey(42)

MalthusJAX Version: 0.2.0
JAX Backend: cpu
Level 1 components loaded successfully


## Part 1: Binary Genomes

Binary genomes represent solutions as bit strings, suitable for combinatorial optimization problems. The `BaseGenome` abstract class defines the interface that `BinaryGenome` implements.

**Key Features**:
- Immutable `@struct.dataclass` structure
- Static factory method `random_init(key, config)`
- Type-safe operations (distance, mutation, etc.)
- Automatic batching via `BinaryPopulation`

In [15]:
# Create binary genome configuration
binary_config = BinaryGenomeConfig(length=10)

# Initialize individual genome
key, subkey = jar.split(key)
genome1 = BinaryGenome.random_init(subkey, binary_config)
print(f"Binary genome 1: {genome1}")

key, subkey = jar.split(key)
genome2 = BinaryGenome.random_init(subkey, binary_config)
print(f"Binary genome 2: {genome2}")

# Demonstrate genome operations
print(f"\nGenome size: {genome1.size}")
print(f"Ones count: {genome1.count_ones()}")

# Distance metrics (showcasing abstract interface implementation)
hamming_dist = genome1.distance(genome2, metric="hamming")
euclidean_dist = genome1.distance(genome2, metric="euclidean")
print(f"\nHamming distance: {hamming_dist}")
print(f"Euclidean distance: {euclidean_dist}")

# Bit manipulation
flipped = genome1.flip_bit(3)
print(f"\nOriginal genome: {genome1}")
print(f"After flipping bit 3: {flipped}")

Binary genome 1: <BinaryGenome(0011111111, len=10)>
Binary genome 2: <BinaryGenome(1111010001, len=10)>

Genome size: 10
Ones count: 8

Hamming distance: 6.0
Euclidean distance: 2.4494898319244385

Original genome: <BinaryGenome(0011111111, len=10)>
After flipping bit 3: <BinaryGenome(0010111111, len=10)>


In [16]:
# Create a population using BasePopulation interface
population_size = 20
key, subkey = jar.split(key)
population = BinaryPopulation.init_random(subkey, binary_config, population_size)

print(f"Population created with {len(population)} individuals")
print(f"Genes shape: {population.genes.bits.shape}")
print(f"Fitness shape: {population.fitness.shape}")

# Population indexing and slicing
print(f"\nFirst individual: {population[0]}")
print(f"Last individual: {population[-1]}")

# Advanced slicing
binary_subset = population[5:10]
print(f"\nSubset size: {len(binary_subset)}")
print(f"Subset genes shape: {binary_subset.genes.bits.shape}")

# Population-level operations
binary_distances = population.distance_matrix(metric="hamming")
print(f"\nDistance matrix shape: {binary_distances.shape}")
print(f"Average pairwise distance: {jnp.mean(binary_distances):.2f}")

Population created with 20 individuals
Genes shape: (20, 10)
Fitness shape: (20,)

First individual: <BinaryGenome(0010111100, len=10)>
Last individual: <BinaryGenome(0000111001, len=10)>

Subset size: 5
Subset genes shape: (5, 10)

Distance matrix shape: (20, 20)
Average pairwise distance: 4.71


In [17]:
# Demonstrate BaseEvaluator with BinarySumEvaluator
# Evaluators are generic: BaseEvaluator[G, C, D] where G is genome type

# OneMax problem: maximize number of 1s
onemax_config = BinarySumConfig(maximize=True)
onemax_evaluator = BinarySumEvaluator(config=onemax_config, data=None)

# Evaluate single genome
fitness = onemax_evaluator.evaluate(genome1)
print(f"Single genome fitness: {fitness}")

# Evaluate population using automatic vectorization
fitness_scores = onemax_evaluator.evaluate_population(population)
print(f"\nPopulation fitness scores shape: {fitness_scores.fitness.shape}")
print(f"Best fitness: {jnp.max(fitness_scores.fitness)}")
print(f"Average fitness: {jnp.mean(fitness_scores.fitness):.2f}")
print(f"Worst fitness: {jnp.min(fitness_scores.fitness)}")

# Find best individual
best_idx = jnp.argmax(fitness_scores.fitness)
best_genome = fitness_scores[best_idx]
print(f"\nBest genome index: {best_idx}")
print(f"Best genome: {best_genome.genes}")

# Demonstrate evaluator flexibility with different problem
# Knapsack problem: maximize value subject to weight constraint
n_items = 20
key, subkey1, subkey2 = jar.split(key, 3)
weights = jar.uniform(subkey1, (n_items,), minval=1.0, maxval=10.0)
values = jar.uniform(subkey2, (n_items,), minval=5.0, maxval=50.0)
capacity = jnp.sum(weights) * 0.5

knapsack_config = KnapsackConfig(
    weights=weights,
    values=values,
    capacity=capacity,
    maximize=True
)
knapsack_evaluator = KnapsackEvaluator(config=knapsack_config, data=None)

# Create population for knapsack
knapsack_genome_config = BinaryGenomeConfig(length=n_items)
key, subkey = jar.split(key)
knapsack_population = BinaryPopulation.init_random(subkey, knapsack_genome_config, population_size)

# Evaluate knapsack problem
knapsack_fitness = knapsack_evaluator.evaluate_population(knapsack_population)
best_knapsack_idx = jnp.argmax(knapsack_fitness.fitness)
best_knapsack_fitness = knapsack_fitness.fitness[best_knapsack_idx]
best_knapsack_genome = knapsack_fitness[best_knapsack_idx]

print(f"\nKnapsack problem:")
print(f"Capacity: {capacity:.1f}")
print(f"Best value: {best_knapsack_fitness:.2f}")
print(f"Best solution: {best_knapsack_genome.genes}")

Single genome fitness: 8.0

Population fitness scores shape: (20,)
Best fitness: 7.0
Average fitness: 4.70
Worst fitness: 2.0

Best genome index: 1
Best genome: <BinaryGenome(0111101011, len=10)>

Knapsack problem:
Capacity: 51.7
Best value: 274.53
Best solution: <BinaryGenome(1010001010..., len=20)>


## Part 2: Real-Valued Genomes

Real-valued genomes represent solutions in continuous spaces, ideal for numerical optimization. The same `BaseGenome` interface is implemented with domain-specific operations.

**Key Features**:
- Bounded continuous values
- Distance metrics (Euclidean, Manhattan)
- Normalization and clamping operations
- Seamless integration with continuous optimization problems

In [18]:
# Create real-valued genome configuration
real_config = RealGenomeConfig(
    length=5,
    bounds=(-5.0, 5.0)
)

# Initialize genomes
key, subkey1, subkey2 = jar.split(key, 3)
real_genome1 = RealGenome.random_init(subkey1, real_config)
real_genome2 = RealGenome.random_init(subkey2, real_config)

print("Real-valued genomes:")
print(f"Genome 1: {real_genome1}")
print(f"Genome 2: {real_genome2}")

# Distance metrics
euclidean_dist = real_genome1.distance(real_genome2, metric="euclidean")
manhattan_dist = real_genome1.distance(real_genome2, metric="manhattan")
print(f"\nEuclidean distance: {euclidean_dist:.4f}")
print(f"Manhattan distance: {manhattan_dist:.4f}")

# Demonstrate operations
print(f"\nOriginal values: {real_genome1.values}")
normalized = real_genome1.normalize()
print(f"Normalized: {normalized.values}")


Real-valued genomes:
Genome 1: <RealGenome([2.928, -3.512, -0.374, 4.013, -2.960], len=5)>
Genome 2: <RealGenome([3.074, 2.066, -0.097, 3.838, -2.932], len=5)>

Euclidean distance: 5.5890
Manhattan distance: 6.2025

Original values: [ 2.9283488  -3.5115457  -0.37373662  4.0129232  -2.9601252 ]
Normalized: [ 0.4321762  -0.51824653 -0.05515739  0.5922416  -0.4368659 ]


In [6]:
# Create real-valued population
key, subkey = jar.split(key)
real_population = RealPopulation.init_random(subkey, real_config, population_size)

print(f"Real population size: {len(real_population)}")
print(f"Genes shape: {real_population.genes.values.shape}")

# Demonstrate Sphere function evaluation (minimize sum of squares)
sphere_config = SphereConfig(maximize=False)  # Minimization
sphere_evaluator = SphereEvaluator(config=sphere_config, data=None)

sphere_fitness = sphere_evaluator.evaluate_population(real_population)
best_sphere_idx = jnp.argmax(sphere_fitness.fitness)  # Max because we negate for minimization
best_sphere = sphere_fitness.fitness[best_sphere_idx]
best_sphere_genome = sphere_fitness[best_sphere_idx]

print(f"\nSphere function evaluation:")
print(f"Best fitness (negated): {best_sphere:.4f}")
print(f"Best genome: {best_sphere_genome.genes}")

# Demonstrate Griewank function (multimodal optimization)
griewank_config = RealGenomeConfig(length=10, bounds=(-600.0, 600.0))
key, subkey = jar.split(key)
griewank_population = RealPopulation.init_random(subkey, griewank_config, population_size)

griewank_evaluator = GriewankEvaluator(config=GriewankConfig(maximize=False), data=None)
griewank_fitness = griewank_evaluator.evaluate_population(griewank_population)

best_griewank = jnp.max(griewank_fitness.fitness)
print(f"\nGriewank function evaluation:")
print(f"Best fitness (negated): {best_griewank:.4f}")
print(f"Population fitness range: [{jnp.min(griewank_fitness.fitness):.2f}, {jnp.max(griewank_fitness.fitness):.2f}]")

Real population size: 20
Genes shape: (20, 5)

Sphere function evaluation:
Best fitness (negated): -2.0175
Best genome: <RealGenome([0.907, 0.750, -0.775, 0.059, -0.164], len=5)>

Griewank function evaluation:
Best fitness (negated): -166.3411
Population fitness range: [-465.15, -166.34]


## Part 3: Categorical Genomes

Categorical genomes represent discrete choices, useful for architecture search, hyperparameter optimization, and combinatorial problems with non-binary choices.

**Key Features**:
- Integer-valued categories with configurable ranges
- Support for variable categories per position
- Permutation support for ordering problems
- Generic interface compatible with all Level 2+ operators

In [7]:
# Create categorical genome configuration
cat_config = CategoricalGenomeConfig(
    length=8,
    num_categories=5  # Each position can be 0-4
)

# Initialize genomes
key, subkey1, subkey2 = jar.split(key, 3)
cat_genome1 = CategoricalGenome.random_init(subkey1, cat_config)
cat_genome2 = CategoricalGenome.random_init(subkey2, cat_config)

print("Categorical genomes:")
print(f"Genome 1: {cat_genome1}")
print(f"Genome 2: {cat_genome2}")

# Distance metric
distance = cat_genome1.distance(cat_genome2)
print(f"\nHamming distance: {distance}")

# Demonstrate swapping positions
swapped = cat_genome1.swap_positions(2, 5)
print(f"\nOriginal: {cat_genome1}")
print(f"After swapping positions 2 and 5: {swapped}")

# Create permutation genome (useful for TSP, scheduling)
perm_config = CategoricalGenomeConfig(length=6, num_categories=6)
key, subkey = jar.split(key)
perm_genome = CategoricalGenome.random_init(subkey, perm_config)
# Ensure it's a valid permutation
permutation = perm_genome.replace(categories=jar.permutation(subkey, jnp.arange(6)))
print(f"\nPermutation genome: {permutation}")

# Validate categories are within bounds
print(f"\nCategory validation:")
print(f"Min category: {jnp.min(cat_genome1.categories)}")
print(f"Max category: {jnp.max(cat_genome1.categories)}")
print(f"All valid: {jnp.all((cat_genome1.categories >= 0) & (cat_genome1.categories < cat_config.num_categories))}")

Categorical genomes:
Genome 1: <CategoricalGenome([4, 0, 4, 1, 4, 4, 3, 2], len=8)>
Genome 2: <CategoricalGenome([0, 1, 0, 2, 4, 3, 4, 0], len=8)>

Hamming distance: 7.0

Original: <CategoricalGenome([4, 0, 4, 1, 4, 4, 3, 2], len=8)>
After swapping positions 2 and 5: <CategoricalGenome([4, 0, 4, 1, 4, 4, 3, 2], len=8)>

Permutation genome: <CategoricalGenome([1, 0, 3, 5, 2, 4], len=6)>

Category validation:
Min category: 0
Max category: 4
All valid: True


## 🟢 Linear Genomes: Symbolic Regression & Genetic Programming

Linear genomes represent **computational DAGs (Directed Acyclic Graphs)** - perfect for symbolic regression, automatic programming, and expression evolution.

### Key Features:
- **Tree-like structure**: Each instruction builds on previous results
- **Mathematical operations**: ADD, SUB, MUL, DIV with configurable arity
- **Automatic rendering**: Human-readable expression display
- **Auto-correction**: Invalid instruction arguments automatically fixed

In [8]:
# Create Linear Genomes for Genetic Programming
print("🟢 Linear Genome Demonstration")
print("="*50)

# Configuration for linear genomes (computational DAGs)
linear_config = mjx.LinearGenomeConfig(
    length=6,           # Number of instructions
    num_inputs=2,       # x_0, x_1 input variables
    num_ops=4,         # ADD, SUB, MUL, DIV operations
    max_arity=3        # Binary operations
)

key, subkey = jar.split(key)

# Create a linear genome
linear_genome = mjx.LinearGenome.random_init(subkey, linear_config)
print(f"✓ Random linear genome: {linear_genome}")
print(f"  - Length: {len(linear_genome.ops)}")
print(f"  - Operations shape: {linear_genome.ops.shape}")
print(f"  - Arguments shape: {linear_genome.args.shape}")

# Display the genome as readable expressions
print(f"\n📋 Genome as mathematical expressions:")
print(linear_genome.render(config=linear_config))

# Create linear genome population for evaluation
key, subkey = jar.split(key)
linear_population = mjx.LinearPopulation.init_random(subkey, linear_config, size=20)
print(f"\n✓ Created linear population: {len(linear_population)} genomes")

# Demonstrate evaluation with LinearGP
print(f"\n🧮 Linear GP Evaluation:")
# Create some sample data for evaluation
key, subkey = jar.split(key)
X = jar.normal(subkey, (10, 2))  # 10 samples, 2 features
y = X[:, 0]**2 + X[:, 1] - 1.0   # Target: x^2 + y - 1

# Setup evaluator with regression data
linear_evaluator = mjx.LinearGPEvaluator(config=linear_config, data=(X, y))
data = (X, y)  # Package X and y as tuple for the evaluator

# Evaluate each genome and get the best instruction fitness from each
linear_fitness_scores = []
for genome in linear_population:
    instruction_fitnesses = linear_evaluator.evaluate(genome)
    best_instruction_fitness = linear_evaluator.get_best_instruction_fitness(instruction_fitnesses)
    linear_fitness_scores.append(float(best_instruction_fitness))

best_linear_fitness = max(linear_fitness_scores)
best_linear_idx = linear_fitness_scores.index(best_linear_fitness)
best_linear_genome = linear_population[best_linear_idx]

print(f"  - Best fitness: {best_linear_fitness:.6f}")
print(f"  - Target function: x^2 + y - 1")
print(f"\n🏆 Best evolved expression:")
print(best_linear_genome.render(config=linear_config))

print("\n" + "="*50)

🟢 Linear Genome Demonstration
✓ Random linear genome: <LinearGenome(L=(6,))>
  - Length: 6
  - Operations shape: (6,)
  - Arguments shape: (6, 3)

📋 Genome as mathematical expressions:
Row  | Expression                     | Raw
--------------------------------------------------
0    | v_0 = OP_3(x_1, x_1, x_0)      | [1 1 0]
1    | v_1 = OP_2(x_1, x_0, x_0)      | [1 0 0]
2    | v_2 = OP_1(x_0, x_1, x_0)      | [0 1 0]
3    | v_3 = OP_0(v_1, x_0, v_2)      | [3 0 4]
4    | v_4 = OP_3(v_2, v_1, x_1)      | [4 3 1]
5    | v_5 = OP_1(x_0, v_3, v_1)      | [0 5 3]

✓ Created linear population: 20 genomes

🧮 Linear GP Evaluation:
  - Best fitness: -0.492780
  - Target function: x^2 + y - 1

🏆 Best evolved expression:
Row  | Expression                     | Raw
--------------------------------------------------
0    | v_0 = OP_2(x_0, x_1, x_0)      | [0 1 0]
1    | v_1 = OP_0(x_0, x_0, x_0)      | [0 0 0]
2    | v_2 = OP_1(x_0, x_0, v_0)      | [0 0 2]
3    | v_3 = OP_0(v_2, v_2, v_1)      

## Part 4: Linear Genetic Programming Genomes

Linear genomes represent executable programs as sequences of instructions, enabling evolution of algorithms and computational structures.

**Key Features**:
- Instruction-based representation
- Register-based computation model
- Configurable operation sets and arity
- Direct evaluation via program execution

In [9]:
# Create sample regression dataset
key, subkey1, subkey2 = jar.split(key, 3)
X = jar.normal(subkey1, (50, 3))  # 50 samples, 3 features
true_coeffs = jnp.array([2.0, -1.5, 0.8])
y = X @ true_coeffs + jar.normal(subkey2, (50,)) * 0.1

# Configure linear genome
linear_config = LinearGenomeConfig(
    length=30,  # Number of instructions
    num_inputs=3,  # Number of input features
    num_ops=10,  # Number of available operations
    max_arity=3,  # Maximum operands per instruction
)

# Create linear genome
key, subkey = jar.split(key)
linear_genome = LinearGenome.random_init(subkey, linear_config)

print(f"Configuration: {linear_config.num_inputs} inputs, {linear_config.num_ops} ops, max arity {linear_config.max_arity}")

# Evaluate single genome
linear_evaluator = LinearGPEvaluator(config=linear_config, data=(X, y))
fitness = linear_evaluator.evaluate(linear_genome)
print(f"\nGenome fitness (negative MSE): {fitness:.4f}")

Configuration: 3 inputs, 10 ops, max arity 3

Genome fitness (negative MSE): -4.6955


In [10]:
# Create linear population
key, subkey = jar.split(key)
linear_population = LinearPopulation.init_random(subkey, linear_config, size=50)

print(f"Linear population size: {len(linear_population)}")

# Evaluate population
evaluated_pop = linear_evaluator.evaluate_population(linear_population)
linear_fitness_scores = evaluated_pop.fitness

best_linear_idx = jnp.argmax(linear_fitness_scores)
best_linear_fitness = linear_fitness_scores[best_linear_idx]
best_linear_genome = evaluated_pop[best_linear_idx].genes

print(f"\nLinear GP evaluation results:")
print(f"Best fitness (negative MSE): {best_linear_fitness:.4f}")
#print(f"Best program length: {len(best_linear_genome)} instructions")
print(f"Fitness distribution: mean={jnp.mean(linear_fitness_scores):.4f}, std={jnp.std(linear_fitness_scores):.4f}")

# Demonstrate instruction analysis
instruction_fitnesses = []
for i in range(min(5, len(linear_population))):
    genome = linear_population[i]
    fit = linear_evaluator.evaluate(genome)
    instruction_fitnesses.append(fit)

print(f"\nSample of individual fitnesses:")
for i, fit in enumerate(instruction_fitnesses):
    print(f"  Individual {i}: {fit:.4f}")

Linear population size: 50

Linear GP evaluation results:
Best fitness (negative MSE): -1.7726
Fitness distribution: mean=-6.5226, std=1.9997

Sample of individual fitnesses:
  Individual 0: -7.2011
  Individual 1: -4.7756
  Individual 2: -8.7154
  Individual 3: -6.5662
  Individual 4: -9.3773


## Summary: Abstract Class Architecture Benefits

This notebook demonstrated how MalthusJAX's Level 1 abstract classes enable:

### 1. **Type Safety Through Generics**
- `BaseGenome`: Common interface for all genome types
- `BasePopulation[G]`: Type-safe population containers parametrized by genome type
- `BaseEvaluator[G, C, D]`: Generic evaluators accepting any genome type

### 2. **Pluggable Components**
Different genome types can be used interchangeably with:
- The same operator interfaces (Level 2)
- The same engine implementations (Level 3)
- Custom fitness functions following the evaluator protocol

### 3. **JAX Optimization**
All components are:
- Immutable `@struct.dataclass` structures (JIT-friendly)
- Vectorized through `jax.vmap()` (GPU-accelerated)
- Configuration-driven (static typing for compilation)

### 4. **Extensibility**
New genome types can be added by implementing:
- `BaseGenome` interface (random_init, distance, etc.)
- `BasePopulation` for vectorized operations
- Custom evaluators following `BaseEvaluator[NewGenome, Config, Data]` pattern

### Performance Characteristics
- **Binary genomes**: Fast bit operations, efficient packing
- **Real genomes**: Native floating-point operations, vectorized math
- **Categorical genomes**: Integer indexing, minimal memory overhead
- **Linear genomes**: Program execution overhead, flexible expressiveness

### Next Steps
- **Level 2**: Genetic operators (mutation, crossover, selection) working with all genome types
- **Level 3**: Evolution engines orchestrating complete evolutionary algorithms
- **Level 4**: Advanced features (multi-objective, constraints, adaptive parameters)